# [2] Regime Adaptive Strategy

This strategy simulates the `sandbox/02_adaptive_settings.yaml` settings. It modifies the underlying risk aversion and turnover lambda parameters of the convex optimizer dynamically based on the current market regime (Bull, Bear, Recovery, Crisis).

Tested strictly over the out-of-sample 2013-2026 walk-forward period.

In [6]:
import sys
import os
from pathlib import Path

# Ensure project root is accessible regardless of where the kernel started
current_dir = Path.cwd()
if current_dir.name == 'sandbox':
    root_dir = str(current_dir.parent)
else:
    root_dir = str(current_dir)

if root_dir not in sys.path:
    sys.path.append(root_dir)

os.chdir(root_dir)

from sandbox.run_backtest import execute_backtest
from sandbox.regime_param_search import run_regime_param_search
from sandbox.viz import stitch_and_display_results
print('Setup complete. Root:', root_dir)

Setup complete. Root: d:\Projects\SystematicPortfolioEngine


In [7]:
battery = 'sandbox/02_adaptive_settings.yaml'

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1  |  Regime Parameter Discovery
# Sweeps all (risk_aversion  x  turnover_lambda) combos defined in the
# search_grid section of 02_adaptive_settings.yaml.
#
# For every combo the full walk-forward simulation is run and each day's
# return is tagged with its market regime. Per-regime Sharpe is computed
# and the winning params per regime are returned.
# ─────────────────────────────────────────────────────────────────────────────
best_params = run_regime_param_search(battery)
print('\nDiscovered best params:', best_params)

🔍 Grid search: 6 risk_aversion × 4 lambda = 24 combos
📦 Loading precomputed data...

🏃 Running 24 simulations...

  [  1/24]  risk_aversion=0.5  lambda=0.002  ... ✓
  [  2/24]  risk_aversion=0.5  lambda=0.005  ... ✓
  [  3/24]  risk_aversion=0.5  lambda=0.007  ... ✓
  [  4/24]  risk_aversion=0.5  lambda=0.010  ... ✓
  [  5/24]  risk_aversion=1.0  lambda=0.002  ... ✓
  [  6/24]  risk_aversion=1.0  lambda=0.005  ... ✓
  [  7/24]  risk_aversion=1.0  lambda=0.007  ... ✓
  [  8/24]  risk_aversion=1.0  lambda=0.010  ... ✓
  [  9/24]  risk_aversion=1.5  lambda=0.002  ... ✓
  [ 10/24]  risk_aversion=1.5  lambda=0.005  ... ✓
  [ 11/24]  risk_aversion=1.5  lambda=0.007  ... ✓
  [ 12/24]  risk_aversion=1.5  lambda=0.010  ... ✓
  [ 13/24]  risk_aversion=2.0  lambda=0.002  ... ✓
  [ 14/24]  risk_aversion=2.0  lambda=0.005  ... ✓
  [ 15/24]  risk_aversion=2.0  lambda=0.007  ... ✓
  [ 16/24]  risk_aversion=2.0  lambda=0.010  ... ✓
  [ 17/24]  risk_aversion=2.5  lambda=0.002  ... ✓
  [ 18/24]  risk_av


Discovered best params: {'recovery': {'risk_aversion': 3.0, 'turnover_lambda': 0.005}}


In [8]:
# STEP 2  |  Adaptive Backtest using discovered regime parameters
# Writes a patched config to sandbox/02_adaptive_settings_optimized.yaml
# (fully within the sandbox folder — original YAML and production unchanged).
import yaml

with open(battery) as f:
    live_config = yaml.safe_load(f)

# Patch discovered best params into the in-memory config
for regime, params in best_params.items():
    live_config.setdefault('adaptive_optimizer', {}).setdefault('regimes', {})
    live_config['adaptive_optimizer']['regimes'].setdefault(regime, {}).update(params)

# Save to sandbox-contained optimized config (inspectable, never overwrites original)
optimized_path = 'sandbox/02_adaptive_settings_optimized.yaml'
with open(optimized_path, 'w') as f:
    yaml.dump(live_config, f, default_flow_style=False, sort_keys=False)
print(f'Optimized config saved to: {optimized_path}')
print('Patched regime params:')
for r, p in live_config['adaptive_optimizer']['regimes'].items():
    print(f'  {r}: risk_aversion={p["risk_aversion"]}, lambda={p["turnover_lambda"]}')

# Run adaptive backtest with the optimized sandbox config
execute_backtest(battery_path=optimized_path)


Optimized config saved to: sandbox/02_adaptive_settings_optimized.yaml
Patched regime params:
  bull: risk_aversion=1.0, lambda=0.007
  recovery: risk_aversion=3.0, lambda=0.005
  bear: risk_aversion=2.5, lambda=0.002
  crisis: risk_aversion=0.5, lambda=0.01
🔋 Plug-and-Play Battery Loaded: 02_adaptive_settings_optimized.yaml

🚀 Initiating walk-forward backtest simulator...
[sandbox] Loading precomputed data globally...
🌍 [Adaptive Mode: ON] Loading precomputed regimes...

[1] Testing OOS: 2013-01-01 to 2013-12-31
Params (Battery): Lambda=0.005 | Risk Aversion=1.5 | MaxWt=0.05
[1] Complete. Final NAV: $1,406,039.82

[2] Testing OOS: 2014-01-01 to 2014-12-31
Params (Battery): Lambda=0.005 | Risk Aversion=1.5 | MaxWt=0.05
[2] Complete. Final NAV: $1,158,953.37

[3] Testing OOS: 2015-01-01 to 2015-12-31
Params (Battery): Lambda=0.005 | Risk Aversion=1.5 | MaxWt=0.05
[3] Complete. Final NAV: $1,022,543.51

[4] Testing OOS: 2016-01-01 to 2016-12-31
Params (Battery): Lambda=0.005 | Risk Avers

In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 3  |  Equity Curve
# ─────────────────────────────────────────────────────────────────────────────
stitch_and_display_results(results_dir='sandbox/results')

🔋 WALK-FORWARD OOS PERFORMANCE METRICS
Total Return:         68.00%
Annualized Return:    84.96%
Annualized Vol:       121.77%
Sharpe Ratio:         0.70
Max Drawdown:         -36.76%
